# Patch-Level Binary Classification Demo

**Backbone:** MLP on synthetic features (mimicking ResNet-152 extracted features).
In real GWHD, replace features with `resnet152(patches)`.

| Concept | Geospatial | This demo |
|---------|------------|----------|
| $s$ | (lat, lon) | (patch_row, patch_col) |
| $t$ | time step | image index |
| $Y(t,s)$ | temperature | wheat head present (0/1) |
| $\Phi(s)$ | spatial eigenfunctions | positional bias pattern |
| $\Sigma$ | station correlation | patch correlation |

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import time
from torch.utils.tensorboard import SummaryWriter

from spatial_adapter import SpatialNeuralAdapter, SpatialBasisLearner, TrendModel
from spatial_adapter.models.spatial_adapter import (
    SpatialNeuralAdapterConfig, ADMMConfig, TrainingConfig, BasisConfig,
)
from spatial_adapter.data.synthetic_patch_binary import get_synthetic_patch_dataloader_and_val
from spatial_adapter.metrics import compute_binary_metrics
from spatial_adapter.cpp_extensions import spatial_utils

torch.manual_seed(42); np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Generate Synthetic Patch Data

In [ ]:
GRID_H, GRID_W = 16, 16
N = GRID_H * GRID_W   # 256
T = 500
P = 64
K_TRUE = 2

train_loader, val_cont, val_y, locs, true_phi, train_prob, val_prob = get_synthetic_patch_dataloader_and_val(
    n_images=T, grid_h=GRID_H, grid_w=GRID_W, feature_dim=P,
    n_basis=K_TRUE, ar_coeff=0.8, signal_std=5.0, noise_std=0.3,
    train_ratio=0.8, batch_size=32, seed=42,
)
print(f'Train: {len(train_loader.dataset)}, Val: {val_cont.shape[0]}, N={N}, P={P}')
print(f'Val positive rate: {val_y.mean():.3f}')

In [ ]:
# Visualize: true basis + sample "images" (probability = original, binary = labels)
n_show = 3
fig, axes = plt.subplots(3, n_show, figsize=(5*n_show, 12))

# Row 1: true spatial basis (pad with avg positive rate if n_show > K_TRUE)
for k in range(min(K_TRUE, n_show)):
    im = axes[0, k].imshow(true_phi[:, k].reshape(GRID_H, GRID_W), cmap='RdBu_r', origin='lower')
    axes[0, k].set_title(f'True $\phi_{k+1}(s)$'); plt.colorbar(im, ax=axes[0, k], shrink=0.8)
if n_show > K_TRUE:
    im = axes[0, K_TRUE].imshow(val_y.mean(0).numpy().reshape(GRID_H, GRID_W), cmap='YlOrRd', origin='lower')
    axes[0, K_TRUE].set_title('Avg positive rate'); plt.colorbar(im, ax=axes[0, K_TRUE], shrink=0.8)

# Row 2: "original images" (underlying probability map)
for j in range(n_show):
    prob_map = val_prob[j].numpy().reshape(GRID_H, GRID_W)
    im = axes[1, j].imshow(prob_map, cmap='hot', origin='lower', vmin=0, vmax=1)
    axes[1, j].set_title(f'Image {j} (probability)')
    plt.colorbar(im, ax=axes[1, j], shrink=0.8)

# Row 3: binary labels (what the model sees)
for j in range(n_show):
    label_map = val_y[j].numpy().reshape(GRID_H, GRID_W)
    axes[2, j].imshow(label_map, cmap='binary_r', origin='lower', vmin=0, vmax=1)
    axes[2, j].set_title(f'Image {j} (labels)')
    for x in range(GRID_W + 1):
        axes[2, j].axvline(x - 0.5, color='gray', linewidth=0.3)
    for y in range(GRID_H + 1):
        axes[2, j].axhline(y - 0.5, color='gray', linewidth=0.3)

axes[0, 0].set_ylabel('Spatial basis', fontsize=13)
axes[1, 0].set_ylabel('Original (prob)', fontsize=13)
axes[2, 0].set_ylabel('Labels (0/1)', fontsize=13)
plt.suptitle('Synthetic patch data', fontsize=15, y=1.01)
plt.tight_layout(); plt.show()


## 2. Verify TPS on 2D Grid

In [ ]:
Omega = spatial_utils.smoothing_penalty_matrix(locs)
print(f'Omega: {Omega.shape}, symmetric={np.allclose(Omega, Omega.T)}')

## 3. Train Spatial Adapter

In [ ]:
K_ADAPTER = 4
trend = TrendModel(num_continuous_features=P, hidden_layer_sizes=[128, 64], n_locations=N)
basis = SpatialBasisLearner(num_locations=N, latent_dim=K_ADAPTER)

config = SpatialNeuralAdapterConfig(
    task='binary',
    admm=ADMMConfig(rho=5.0, dual_momentum=0.2, max_iters=500, min_outer=100, tol=1e-4),
    training=TrainingConfig(lr_mu=1e-3, batch_size=32, pretrain_epochs=20),
    basis=BasisConfig(),
)
print(f'Trend: {sum(p.numel() for p in trend.parameters()):,} params')
print(f'Basis: {N*K_ADAPTER:,} params')

In [ ]:
writer = SummaryWriter('/tmp/patch_demo')
adapter = SpatialNeuralAdapter(
    trend=trend, basis=basis, train_loader=train_loader,
    val_cont=val_cont, val_y=val_y, locs=locs.astype(np.float32),
    config=config, device=device, writer=writer, tau1=0.1, tau2=0.01,
)

print('Pretraining trend (BCE)...')
adapter.pretrain_trend()
print('Init basis...')
adapter.init_basis_dense()
print('ADMM...')
t0 = time.time()
best_val = adapter.run()
writer.close()
print(f'\nDone in {time.time()-t0:.1f}s, best val acc: {best_val:.4f}')

## 4. Baseline vs Adapter

In [ ]:
adapter.trend.eval(); adapter.basis.eval()
with torch.no_grad():
    logits_b = adapter.trend(val_cont.to(device))
    acc_b, f1_b, auc_b = compute_binary_metrics(val_y, logits_b.cpu())
    logits_a = adapter.reconstruct(val_cont.to(device), val_y.to(device))
    acc_a, f1_a, auc_a = compute_binary_metrics(val_y, logits_a.cpu())

print(f"{'Model':<30} {'Acc':>8} {'F1':>8} {'AUC':>8}")
print('-'*54)
print(f"{'MLP (baseline)':<30} {acc_b:>8.4f} {f1_b:>8.4f} {auc_b:>8.4f}")
print(f"{'MLP + Adapter':<30} {acc_a:>8.4f} {f1_a:>8.4f} {auc_a:>8.4f}")

## 5. Learned $\Phi(s)$

In [ ]:
learned_phi = adapter.basis.basis.detach().cpu().numpy()
fig, axes = plt.subplots(2, max(K_TRUE, K_ADAPTER), figsize=(4*max(K_TRUE, K_ADAPTER), 8))
for k in range(K_TRUE):
    im = axes[0,k].imshow(true_phi[:,k].reshape(GRID_H,GRID_W), cmap='RdBu_r', origin='lower')
    axes[0,k].set_title(f'True $\\phi_{k+1}$'); plt.colorbar(im, ax=axes[0,k], shrink=0.8)
for k in range(K_TRUE, max(K_TRUE, K_ADAPTER)): axes[0,k].axis('off')
for k in range(K_ADAPTER):
    im = axes[1,k].imshow(learned_phi[:,k].reshape(GRID_H,GRID_W), cmap='RdBu_r', origin='lower')
    axes[1,k].set_title(f'Learned $\\hat{{\\phi}}_{k+1}$'); plt.colorbar(im, ax=axes[1,k], shrink=0.8)
axes[0,0].set_ylabel('True', fontsize=14); axes[1,0].set_ylabel('Learned', fontsize=14)
plt.suptitle('Spatial Basis Recovery', fontsize=16, y=1.02)
plt.tight_layout(); plt.show()

alignment = np.abs(true_phi.T @ learned_phi)
print(f'Subspace alignment:\n{np.round(alignment, 3)}')
print(f'Max alignment per true basis: {np.round(alignment.max(axis=1), 4)}')

## 6. Covariance Estimation

In [ ]:
adapter.trend.eval()
with torch.no_grad():
    _, tc, ty = train_loader.dataset.tensors
    logit_y = torch.special.logit(ty.clamp(1e-7, 1-1e-7))
    mu = adapter.trend(tc.to(device)).cpu()
    residuals = (logit_y - mu).numpy()

result = spatial_utils.estimate_covariance(learned_phi, residuals)
eigenvalues = result['eigenvalues']
noise_var = result['noise_var']
est_cov = result['estimated_covariance']

print(f'Eigenvalues: {np.round(eigenvalues[:4], 4)}')
print(f'Noise var: {noise_var:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im = axes[0].imshow(est_cov, cmap='coolwarm', aspect='auto')
axes[0].set_title('$\\hat{\\Sigma}$'); plt.colorbar(im, ax=axes[0])
axes[1].bar(range(len(eigenvalues)), eigenvalues)
axes[1].set_title('Eigenvalue spectrum'); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Prediction Intervals

In [ ]:
from scipy import stats

logit_var = np.diag(est_cov) + noise_var
logit_std = np.sqrt(np.maximum(logit_var, 0))
z = stats.norm.ppf(0.975)

with torch.no_grad():
    lv = adapter.reconstruct(val_cont.to(device), val_y.to(device)).cpu().numpy()

sig = lambda x: 1/(1+np.exp(-np.clip(x,-20,20)))
p_hat = sig(lv)
p_lo = sig(lv - z*logit_std[None,:])
p_hi = sig(lv + z*logit_std[None,:])

# For binary labels, coverage = label falls within [p_lo, p_hi]
# Since Y in {0,1}: covered if (Y=0 and p_lo <= 0) or (Y=1 and p_hi >= 1)
# More practically: the PI covers the predicted probability, not the hard label.
# We report the PI width and whether the interval is informative.
y_val = val_y.numpy()

# Coverage: does the true class have probability within the PI?
# For Y=1: covered if p_hi > 0.5 (model doesn't rule out positive)
# For Y=0: covered if p_lo < 0.5 (model doesn't rule out negative)
covered = np.where(y_val == 1, p_hi >= 0.5, p_lo <= 0.5)
cp = covered.mean()
mpiw = (p_hi - p_lo).mean()

print(f'95% PI: CP={cp:.4f}, MPIW={mpiw:.4f}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
im = axes[0].imshow(logit_std.reshape(GRID_H,GRID_W), cmap='YlOrRd', origin='lower')
axes[0].set_title('Logit std $\\sqrt{v(s)}$'); plt.colorbar(im, ax=axes[0], shrink=0.8)
im = axes[1].imshow((p_hi-p_lo).mean(0).reshape(GRID_H,GRID_W), cmap='YlOrRd', origin='lower')
axes[1].set_title('Avg PI width'); plt.colorbar(im, ax=axes[1], shrink=0.8)
im = axes[2].imshow(covered.mean(0).reshape(GRID_H,GRID_W), cmap='RdYlGn', origin='lower', vmin=0.8, vmax=1)
axes[2].set_title('Coverage'); plt.colorbar(im, ax=axes[2], shrink=0.8)
plt.tight_layout(); plt.show()

## 8. Summary

In [ ]:
print('='*54)
print('PATCH CLASSIFICATION DEMO')
print('='*54)
print(f'Grid: {GRID_H}x{GRID_W}={N}, Images: {T}, K={K_ADAPTER}')
print()
print(f"{'Model':<30} {'Acc':>8} {'F1':>8} {'AUC':>8}")
print('-'*54)
print(f"{'MLP (baseline)':<30} {acc_b:>8.4f} {f1_b:>8.4f} {auc_b:>8.4f}")
print(f"{'MLP + Adapter':<30} {acc_a:>8.4f} {f1_a:>8.4f} {auc_a:>8.4f}")
print()
print(f'Eigenvalues: {np.round(eigenvalues[:3], 4)}')
print(f'Noise var: {noise_var:.4f}')
print(f'95% PI: CP={cp:.4f}, MPIW={mpiw:.4f}')
print(f'Alignment: {np.round(alignment.max(axis=1), 4)}')
print('='*54)